## Embedding Adjustment Notebook

This will contain the model to adjust the embedding space in some way to accurately represent user inputs.

Current plan is to use a Neural Network from PyTorch to altar the values of the embeddings to other embeddings with the exact same shape. Building the network is easy, but figuring out the loss function is harder.

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

## Neural Network

This section contains the Neural Network that adjusts the embeddings. This sets up the architecture and tests forward with just basic inputs.

In [ ]:
# Architecture of the model (can change)
class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()
        self.layer1 = nn.Linear(1024, 1024)
        self.layer2 = nn.Linear(1024, 1024)

    def forward(self, x):
        x = self.layer1(x)
        x = nn.functional.relu(x)
        x = self.layer2(x)
        out = nn.functional.sigmoid(x)
        return out

In [ ]:
# model and fake inputs
model = Net()
inputs = [[1.0] * 1024] * 5
inputs = torch.tensor(inputs)
print(len(inputs), len(inputs[0]))

5 1024


In [15]:
# runs all inputs in model
for n in inputs:
    input = n.detach().clone()
    print(model(n).detach())

tensor([0.5120, 0.5065, 0.4410,  ..., 0.4543, 0.5499, 0.5358])
tensor([0.5120, 0.5065, 0.4410,  ..., 0.4543, 0.5499, 0.5358])
tensor([0.5120, 0.5065, 0.4410,  ..., 0.4543, 0.5499, 0.5358])
tensor([0.5120, 0.5065, 0.4410,  ..., 0.4543, 0.5499, 0.5358])
tensor([0.5120, 0.5065, 0.4410,  ..., 0.4543, 0.5499, 0.5358])


## Loss Function

This contains how we calculate loss. This section does not have code (maybe it will) but is here to guide the user through the thought process behind the loss.

Current idea is to mark objects that should be close together based on predefined clusters, and objects that the user has specified should be close together. User specified should be weighted much more.

Perhaps in the pre-processing, we have a function that takes each data point and assigns them the label which is the cluster they were assign (arbitrary integer identifier for the cluster). Then which of these data points were specified by the user, these elements will have a much higher weight when claculating the loss.

## Data Preparation

This section contains the necessary pre-processing done on the data

In [ ]:
class ClusterDataset(Dataset):
    def __init__(self, inputs, labels, user_specified, mode = 'train'):
        self.mode = mode
        self.user_specified = user_specified
        if self.mode == 'train':
          self.train = inputs.reshape(-1, 1024).float()
          self.train_labels = labels

    def __len__(self):
        return self.train.shape[0]

    def __getitem__(self, idx):
        if self.mode == 'train':
          return {'input': self.train[idx], 'cluster': self.train_labels[idx]}

In [18]:
batch_size = 8
learning_rate = 0.001
epochs = 100

In [20]:
def load_data(inputs, clusters, user_specified):
    train_set = ClusterDataset(inputs, clusters, user_specified)
    train_dataloader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

    return train_dataloader

## NN Training

This will train the network with the loss function defined above, way unfinished

In [ ]:
def train_loop(data):
    inputs, clusters, user_specified = data
    train_dataloader = load_data(inputs, clusters, user_specified)
    init_train_losses = []
    init_train_accuracies = []
    for epoch in range(1,epochs+1):

        #Training phase
        model.train()  #Setting the model to train phase
        train_loss = []
        train_acc = 0.

        for idx, batch in enumerate(train_dataloader):

            inputs = batch['input']
            labels = batch['label']
            outputs = torch.squeeze(model(inputs.float()))

            loss = loss_function(outputs, labels.float())
            loss.backward()
            optim.step()
            train_loss.append(loss.item())
            train_acc += (outputs.round() == labels).sum().item()
            optim.zero_grad()
        train_acc = train_acc / len(train_set)
        
        if epoch%10==0:
            print("Epoch : {}, Train loss: {} , Train Acc: {}, Val loss: {}, Val acc: {}".format(epoch, np.mean(train_loss), train_acc, np.mean(val_loss), val_acc))
            init_train_losses.append(np.mean(train_loss))
            init_train_accuracies.append(train_acc)